In [ ]:
import os
import pandas as pd

def reesamplear_dataset(ruta_txt, ruta_csv_salida):
    """
    Tu función original: Lee, etiqueta, reesamplea (200Hz -> 50Hz) y guarda.
    """
    nombre_archivo = os.path.basename(ruta_txt)
    
    if nombre_archivo.startswith('F'):
        tag = 1
    elif nombre_archivo.startswith('D'):
        tag = 0
    else:
        return None

    columnas_deseadas = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
    
    try:
        df = pd.read_csv(
            ruta_txt, 
            header=None, 
            usecols=[0, 1, 2, 3, 4, 5], 
            names=columnas_deseadas
        )

        # Reesamplear (promedio cada 4 filas)
        df_resampleado = df.groupby(df.index // 4).mean()
        df_resampleado['tag'] = tag
        
        df_resampleado.to_csv(ruta_csv_salida, index=False)
        return df_resampleado
    except Exception as e:
        print(f"Error al procesar {nombre_archivo}: {e}")
        return None

def procesar_carpeta(carpeta_entrada, carpeta_salida):
    """
    Procesa todos los archivos .txt de una carpeta específica.
    """
    os.makedirs(carpeta_salida, exist_ok=True)
    archivos = os.listdir(carpeta_entrada)
    contador = 0

    for archivo in archivos:
        if archivo.endswith('.txt'):
            ruta_txt = os.path.join(carpeta_entrada, archivo)
            nuevo_nombre = archivo.replace('.txt', '.csv')
            ruta_csv_salida = os.path.join(carpeta_salida, nuevo_nombre)
            
            reesamplear_dataset(ruta_txt, ruta_csv_salida)
            contador += 1
    
    return contador

def ejecutar_pipeline_sisfall(ruta_raiz):
    """
    LA FUNCIÓN MAESTRA:
    Entra a 'SisFall_dataset', busca cada subcarpeta y aplica el proceso.
    """
    # Definimos el nombre de la carpeta de salida
    ruta_salida_base = ruta_raiz + "_PROCESADO_50Hz"
    
    if not os.path.exists(ruta_raiz):
        print(f"Error: No se encuentra la carpeta '{ruta_raiz}'")
        return

    # Listar subcarpetas dentro de SisFall_dataset (SA01, SA02, SE01...)
    subcarpetas = [f for f in os.listdir(ruta_raiz) if os.path.isdir(os.path.join(ruta_raiz, f))]
    
    print(f"--- Iniciando Pipeline en {ruta_raiz} ---")
    print(f"Se encontraron {len(subcarpetas)} carpetas de sujetos.")

    total_archivos = 0
    for sub in subcarpetas:
        ruta_in = os.path.join(ruta_raiz, sub)
        ruta_out = os.path.join(ruta_salida_base, sub)
        
        print(f"Procesando sujeto: {sub}...", end="\r")
        procesados = procesar_carpeta(ruta_in, ruta_out)
        total_archivos += procesados

    print(f"\n\n¡Éxito total!")
    print(f"Se crearon {total_archivos} archivos CSV en '{ruta_salida_base}'.")

# --- EJECUCIÓN ---
ejecutar_pipeline_sisfall("SisFall_dataset")